# Hugh Jackman Detector using ViT-B/32
This notebook uses a pre-trained Vision Transformer (ViT-B/32) to detect if Hugh Jackman is in a given image.

## 1. Install Dependencies

In [1]:
%pip install torch torchvision timm face_recognition pillow numpy

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/109.3 MB ? eta -:--:--
   ---------------------------------------- 0.8/109.3 MB 3.1 MB/s eta 0:00:35
    --------------------------------------- 2.4/109.3 MB 5.4 MB/s eta 0:00:20
   - -------------------------------------- 3.9/109.3 MB 6.2 MB/s eta 0:00:17
   - -------------------------------------- 5.0/109.3 MB 5.8 MB/s eta 0:00:18
   -- ------------------------------------- 6.8/109.3 MB 6.4 MB/s eta 0:00:16
   --- ------------------------------------ 8.7/109.3 MB 6.7 MB/s eta 0:00:16
 


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Libraries and Load Model

In [2]:
import torch
import timm
from PIL import Image
import face_recognition
import numpy as np
import torchvision.transforms as T
from numpy.linalg import norm

print(f'PyTorch version: {torch.__version__}')
print(f'Timm version: {timm.__version__}')

# Load the ViT-B/32 model pre-trained on ImageNet-21k and fine-tuned on ImageNet-1k
model = timm.create_model('vit_base_patch32_224_in21k', pretrained=True)
model.eval()

# Get the data configuration for the model
data_config = timm.data.resolve_data_config({}, model=model)
transform = timm.data.create_transform(**data_config)

print('ViT-B/32 model loaded successfully.')

d:\Projects\ImageEngine\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Projects\ImageEngine\venv\Lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


PyTorch version: 2.9.0+cpu
Timm version: 1.0.22


d:\Projects\ImageEngine\venv\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name vit_base_patch32_224_in21k to current vit_base_patch32_224.augreg_in21k.
  model = create_fn(


ViT-B/32 model loaded successfully.


d:\Projects\ImageEngine\venv\Lib\site-packages\huggingface_hub\file_download.py:121: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\khali\.cache\huggingface\hub\models--timm--vit_base_patch32_224.augreg_in21k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## 3. Feature Extraction Function

In [3]:
def get_embedding(face_image):
    # Converts a face image (PIL Image) to a feature embedding using the ViT model.
    # Preprocess the image
    img_tensor = transform(face_image).unsqueeze(0) # Add batch dimension
    
    # Get the feature embedding (before the classification head)
    with torch.no_grad():
        embedding = model.forward_features(img_tensor)
        # For ViT, we can take the embedding of the [CLS] token
        embedding = embedding[:, 0]
        
    return embedding.numpy().flatten()

## 4. Generate Reference Embedding for Hugh Jackman

In [4]:
REFERENCE_IMAGE_PATH = 'Images/references/104711.jpg'
KNOWN_NAME = 'Hugh Jackman'

try:
    reference_image = face_recognition.load_image_file(REFERENCE_IMAGE_PATH)
    face_locations = face_recognition.face_locations(reference_image)
    
    if face_locations:
        # Assuming the first face found is the correct one
        top, right, bottom, left = face_locations[0]
        reference_face_image = reference_image[top:bottom, left:right]
        reference_face_pil = Image.fromarray(reference_face_image)
        
        # Get the embedding
        reference_embedding = get_embedding(reference_face_pil)
        print(f'Successfully generated embedding for {KNOWN_NAME}.')
    else:
        print(f'Error: No face found in the reference image at {REFERENCE_IMAGE_PATH}')
        reference_embedding = None

except FileNotFoundError:
    print(f'Error: Reference image not found at {REFERENCE_IMAGE_PATH}')
    reference_embedding = None

Successfully generated embedding for Hugh Jackman.


## 5. Detection Function

In [5]:
def detect_hugh_jackman_in_image(image_path, threshold=0.8):
    """
    Detects if Hugh Jackman is in the given image.
    Returns a list of labels ('Hugh Jackman') if detected, otherwise an empty list.
    """
    if reference_embedding is None:
        print('Cannot perform detection because the reference embedding is not available.')
        return []
    
    try:
        target_image = face_recognition.load_image_file(image_path)
        face_locations = face_recognition.face_locations(target_image)
        
        if not face_locations:
            print(f'No faces found in {image_path}')
            return []
            
        print(f'Found {len(face_locations)} face(s) in {image_path}. Comparing to {KNOWN_NAME}...')
        
        for top, right, bottom, left in face_locations:
            face_image = target_image[top:bottom, left:right]
            face_pil = Image.fromarray(face_image)
            
            # Get embedding for the face
            embedding = get_embedding(face_pil)
            
            # Compare with reference embedding using cosine similarity
            cosine_similarity = np.dot(reference_embedding, embedding) / (norm(reference_embedding) * norm(embedding))
            
            print(f'  - Cosine similarity: {cosine_similarity:.4f}')
            
            if cosine_similarity > threshold:
                print(f'  --> Match found! Detected {KNOWN_NAME}.')
                return [KNOWN_NAME]
        
        print('No match found.')
        return []
        
    except FileNotFoundError:
        print(f'Error: Image not found at {image_path}')
        return []

## 6. Test the Detector

In [7]:
# You can test the detector here. 
# Find an image of Hugh Jackman and an image without him to test.

# Example usage (replace with your image paths):
hugh_jackman_image = 'Images/3899/062770.jpg'
other_image = 'Images/not_3899/017031.jpg'

result1 = detect_hugh_jackman_in_image(hugh_jackman_image)
print(f'Detection result for {hugh_jackman_image}: {result1}')

result2 = detect_hugh_jackman_in_image(other_image)
print(f'Detection result for {other_image}: {result2}')

Found 1 face(s) in Images/3899/062770.jpg. Comparing to Hugh Jackman...
  - Cosine similarity: 0.6916
No match found.
Detection result for Images/3899/062770.jpg: []
Found 1 face(s) in Images/not_3899/017031.jpg. Comparing to Hugh Jackman...
  - Cosine similarity: 0.5899
No match found.
Detection result for Images/not_3899/017031.jpg: []
